# Maize Leaf Disease Classifier — Live API Demo (Colab)

Serve the trained classifier as a public REST API using ngrok, accessible from any smartphone.

## Prerequisites

1. A trained TFLite model (`mobilenetv2_best.tflite`) and metadata (`mobilenetv2_meta.json`)
   stored in Google Drive at `MyDrive/maize-model/`
2. A free ngrok account: https://dashboard.ngrok.com/signup
3. Your ngrok auth token: https://dashboard.ngrok.com/get-started/your-authtoken

## Limitations

- Colab sessions expire after ~8 hours (free) / 12 hours (Pro); URL dies with session
- Suitable for demonstrations only — not production traffic
- The public URL changes each session; re-run Cells 6–7 to get a new URL
- Rate limiting applies: 20 requests/minute per IP (from existing API code)

In [ ]:
# Install Colab-only dependencies (NOT in requirements.txt — demo use only)
!pip install fastapi "uvicorn[standard]" pyngrok python-multipart slowapi pydantic-settings -q
print("Dependencies installed.")

In [ ]:
import os
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Adjust these paths if you saved your model elsewhere in Google Drive
MODEL_DRIVE_PATH = '/content/drive/MyDrive/maize-model/mobilenetv2_best.tflite'
META_DRIVE_PATH  = '/content/drive/MyDrive/maize-model/mobilenetv2_meta.json'

os.makedirs('/content/model_artifacts', exist_ok=True)
shutil.copy(MODEL_DRIVE_PATH, '/content/model_artifacts/mobilenetv2_best.tflite')
shutil.copy(META_DRIVE_PATH,  '/content/model_artifacts/mobilenetv2_meta.json')

print("Model artifacts ready:")
print(os.listdir('/content/model_artifacts'))

In [ ]:
import sys
import os

# Clone the repo if not already present
REPO_PATH = '/content/maize-leaf-classifier'
if not os.path.exists(REPO_PATH):
    os.system(f'git clone https://github.com/YOUR_USERNAME/maize-leaf-classifier.git {REPO_PATH}')
    print(f"Repo cloned to {REPO_PATH}")

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# CRITICAL: Set env vars before importing api.* (pydantic-settings reads at import time)
os.environ.update({
    'MODEL_PATH':      '/content/model_artifacts/mobilenetv2_best.tflite',
    'MODEL_META_PATH': '/content/model_artifacts/mobilenetv2_meta.json',
    'ALLOWED_ORIGINS': '["*"]',
    'DEBUG':           'true',   # enables /docs Swagger UI
})

from api.main import create_app
app = create_app()

print("FastAPI app created.")
print("Available routes:", [r.path for r in app.routes])

In [ ]:
import threading
import time
import requests
import uvicorn

_server_started = False

def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

if not _server_started:
    thread = threading.Thread(target=_run_server, daemon=True)
    thread.start()
    _server_started = True

time.sleep(3)  # Wait for uvicorn startup + TFLite model load (~1-2s)

# Verify the server is healthy
try:
    r = requests.get("http://localhost:8000/health", timeout=10)
    health = r.json()
    print(f"Server status: {health['status']}")
    print(f"Model loaded: {health['model_loaded']}")
    if health.get('status') != 'ok':
        print("WARNING: Model not loaded — check MODEL_PATH and META_PATH in Cell 3-4")
    else:
        print("Server is ready to accept requests.")
except Exception as e:
    print(f"Server not responding: {e}")
    print("Try increasing time.sleep() above or re-running this cell.")

In [ ]:
from pyngrok import ngrok

# Get your free auth token at: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_TOKEN_HERE"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()  # Close any tunnels from a previous run of this cell

tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url

print(f"""
{'='*60}
  Public API URL:   {PUBLIC_URL}
  Predict:          {PUBLIC_URL}/predict
  Health check:     {PUBLIC_URL}/health
  Swagger UI:       {PUBLIC_URL}/docs
{'='*60}

Share the URL or scan the QR code in the next cell.
""")

In [ ]:
!pip install "qrcode[pil]" -q
import qrcode
from IPython.display import display

qr = qrcode.QRCode(
    version=None,
    box_size=6,
    border=2,
    error_correction=qrcode.constants.ERROR_CORRECT_M,
)
qr.add_data(PUBLIC_URL)
qr.make(fit=True)

# Use the app's brand colour (#2d6a4f — matches vite.config.js PWA manifest)
img = qr.make_image(fill_color="#2d6a4f", back_color="white")
img.save('/content/api-qr.png')
display(img)

print(f"\nQR code URL: {PUBLIC_URL}")
print("Scan this QR code with a smartphone to access the classifier.")
print("Or share the URL directly for browser access.")

In [ ]:
import io
import json
import numpy as np
from PIL import Image

def _make_test_image() -> bytes:
    """224x224 synthetic green image for smoke testing."""
    arr = np.zeros((224, 224, 3), dtype=np.uint8)
    arr[:, :, 1] = 128  # green channel
    buf = io.BytesIO()
    Image.fromarray(arr).save(buf, format="JPEG")
    return buf.getvalue()

test_img = _make_test_image()

# Test local endpoint
print("Testing local API (localhost:8000)...")
r_local = requests.post(
    "http://localhost:8000/predict",
    files={"file": ("test.jpg", test_img, "image/jpeg")},
    timeout=15,
)
print(f"  Status: {r_local.status_code}")
print(f"  Response: {json.dumps(r_local.json(), indent=2)}")

# Test public endpoint
print(f"\nTesting public API ({PUBLIC_URL})...")
r_public = requests.post(
    f"{PUBLIC_URL}/predict",
    files={"file": ("test.jpg", test_img, "image/jpeg")},
    timeout=30,
)
print(f"  Status: {r_public.status_code}")
print(f"  Response: {json.dumps(r_public.json(), indent=2)}")

print("""
=====================================
CLEANUP & SESSION MANAGEMENT
=====================================
To stop the ngrok tunnel:
  ngrok.disconnect(tunnel.public_url)

To get a new URL (after disconnecting or session restart):
  Re-run Cells 6 and 7.

The URL expires automatically when this Colab session ends.
uvicorn stops when the runtime process exits (daemon thread).
""")